# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kifayat-Sayed/flyrank-ml-internship1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



**Task type: Ranking / scoring.**

The decision is: **which content items should an editor investigate or refresh first?**

The model would produce a priority score for each content item. Editors could then work down the ranked queue instead of reviewing all content equally.

I chose ranking/scoring because the operational question is about **which items come first**, rather than assigning every item to a fixed category. This matches the lane's practical content-refresh workflow.

In [1]:

from pathlib import Path
import pandas as pd

# 1. Check common local locations first
candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/data/raw/content_refresh_anonymized.csv"),
    Path("/content/content_refresh_anonymized.csv"),
    Path("/content/content_refresh_anonymized (1).csv"),
    Path("/mnt/data/content_refresh_anonymized.csv"),
]

csv_path = next((p for p in candidates if p.is_file()), None)

# 2. If not found locally, try the public dataset and repository
if csv_path is not None:
    data = pd.read_csv(csv_path)
    source = str(csv_path)
else:
    urls = [
        "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv",
        "https://raw.githubusercontent.com/Kifayat-Sayed/flyrank-ml-internship1/main/data/raw/content_refresh_anonymized.csv",
    ]

    errors = []
    data = None
    source = None

    for url in urls:
        try:
            candidate_data = pd.read_csv(url)
            if candidate_data.empty:
                raise ValueError("The CSV is empty.")

            data = candidate_data
            source = url
            break
        except Exception as exc:
            errors.append(f"{url}\n  {type(exc).__name__}: {exc}")

    if data is None:
        raise RuntimeError(
            "Could not load the dataset from any source:\n"
            + "\n".join(errors)
            + "\nUpload the CSV to Colab or check the repository path."
        )

# 3. Validate the columns needed by this notebook
required_cols = [
    "client_id",
    "content_id",
    "days_since_last_update",
    "impressions_90d",
]

missing = [col for col in required_cols if col not in data.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

# 4. Show the result
print(f"Loaded from: {source}")
print(f"Shape: {data.shape}")
print(f"Unique clients: {data['client_id'].nunique():,}")
print(f"Unique content items: {data['content_id'].nunique():,}")
print("Dataset loaded and required columns validated.")
display(data.head())


Loaded from: https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv
Shape: (30000, 53)
Unique clients: 32
Unique content items: 30,000
Dataset loaded and required columns validated.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target I would use for a production model is an **observed future outcome**, not the existing rule-derived `trend_direction` or `trend_pct`.

**Proposed target:** `declines_next_30d` — a binary outcome indicating whether the content item experiences a predefined decline in the **following 30-day observation window**.

The key point is that the label must be measured from a later window than the features used to make the prediction. The current starter CSV does not contain a later 30-day window after its snapshot, so I cannot honestly calculate this final target from this file alone.

For this framing exercise, I therefore **sketch the target column** and explicitly leave it unfilled. I will not use `trend_direction`, `trend_pct`, or `is_declining_label` as features because they are derived from the same current trend logic and would turn the model into a reproduction of an existing rule.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sketch the target column without pretending that the starter snapshot contains future outcomes.
target_sketch = data[["content_id", "client_id"]].head(10).copy()
target_sketch["declines_next_30d"] = pd.Series([pd.NA] * len(target_sketch), dtype="boolean")

print("Target sketch: one future observed outcome per content item")
display(target_sketch)

print("\nExisting current trend fields are present in the snapshot:")
print(data[["trend_direction", "trend_pct"]].head().to_string(index=False))

Target sketch: one future observed outcome per content item


,content_id,client_id,declines_next_30d
0,content_304f48230142,client_f369cb89fc,<NA>
1,content_a1fb4e703a9e,client_4e07408562,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,<NA>
3,content_331d6c4de07b,client_19581e27de,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,<NA>
5,content_d4084a4bc775,client_f369cb89fc,<NA>
6,content_9a34b442b552,client_8722616204,<NA>
7,content_a63219c6e95a,client_19581e27de,<NA>
8,content_5e6c160719bc,client_6208ef0f77,<NA>
9,content_c27558df2b0c,client_19581e27de,<NA>



Existing current trend fields are present in the snapshot:
trend_direction  trend_pct
           down      -41.4
           down      -57.7
           down      -60.9
         stable      -13.8
           down      -34.7


## 3. Success metric

**Primary metric: Precision@50.**

A good ranking should put a high proportion of genuinely declining content into the first 50 recommendations.

I chose Precision@50 because the editor has limited time and is likely to act on a small top-ranked queue first. The metric therefore connects directly to the decision: **of the first 50 items recommended for attention, how many actually show the future outcome we care about?**

The exact target prevalence and Precision@50 can only be measured once a future outcome window is available. I will define the metric before model training rather than choosing it after seeing model results.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show why the top-K metric is tied to a finite editorial queue.
K = 50
print(f"Primary evaluation metric: Precision@{K}")
print(f"Interpretation: among the top {K} ranked content items, measure the share with the observed future decline outcome.")
print("The future outcome is not available in this 30,000-row snapshot, so no performance value is claimed here.")


Primary evaluation metric: Precision@50
Interpretation: among the top 50 ranked content items, measure the share with the observed future decline outcome.
The future outcome is not available in this 30,000-row snapshot, so no performance value is claimed here.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item.**

The starter data has 30,000 rows and 30,000 unique `content_id` values. Each row contains content-level characteristics and trailing performance signals. `client_id` identifies the client and is useful for grouping/splitting, but it is not itself a predictive feature.

For the framing stage, the lane slice below keeps content identity plus representative signals that an editor/model could use. Outcome-derived fields are shown only for understanding the dataset and are not proposed as features.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_columns = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "search_volume",
    "impressions_last_30d",
    "clicks_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "avg_position",
    "content_age_days",
]

lane_slice = data[lane_columns].head(8).copy()

print("Rows:", len(data))
print("Unique content items:", data["content_id"].nunique())
print("One row = one content item")
display(lane_slice)

Rows: 30000
Unique content items: 30000
One row = one content item


,content_id,client_id,content_type,main_intent,search_volume,impressions_last_30d,clicks_last_30d,impressions_prev_30d,clicks_prev_30d,avg_position,content_age_days
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,10.0,578,2,987,13,10.6,187
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,90.0,2501,2,5915,1,20.3,445
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,0.0,2382,1,6089,3,36.5,141
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,10.0,3626,22,4206,17,6.2,463
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,0.0,4211,10,6452,2,44.0,263
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,720.0,617,0,1009,1,8.5,147
6,content_9a34b442b552,client_8722616204,keyword article,informational,0.0,1,0,13,0,7.0,90
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,590.0,636,1,632,0,21.2,445


## 5. Why ML beats a fixed rule here

A fixed rule could say something like **“refresh every page whose trend falls below a chosen threshold.”** That is easy to write, but it uses one hand-written condition and does not naturally combine the many signals available for each content item.

The prioritization problem can involve search demand, competition, content type and intent, age/freshness, recent versus previous performance, impressions, clicks, and search position. These signals can interact, and their usefulness may vary across content items and clients.

ML earns its place if a learned ranking can combine these signals and place more **future-observed declining items** in the limited top-50 editorial queue than a simple fixed rule.

This is a **decision-support** use of ML: the model recommends an order for human review; it does not automatically decide which content must be changed.

I will only claim an ML advantage after evaluating it against a fixed baseline on an appropriately separated future outcome window.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick check of the breadth of signals available for a learned ranking.
candidate_feature_groups = {
    "demand": ["search_volume", "competition", "cpc"],
    "content": ["content_type", "main_intent", "word_count", "content_age_days", "days_since_last_update"],
    "recent_performance": ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d"],
    "previous_performance": ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"],
    "search_performance": ["avg_position", "impressions_90d", "clicks_90d"],
}

for group, cols in candidate_feature_groups.items():
    available = [c for c in cols if c in data.columns]
    print(f"{group}: {len(available)} signals -> {', '.join(available)}")

print("\nLeakage check: these current trend-derived fields will not be used as features:")
print(["trend_direction", "trend_pct", "is_declining_label"])


demand: 3 signals -> search_volume, competition, cpc
content: 5 signals -> content_type, main_intent, word_count, content_age_days, days_since_last_update
recent_performance: 3 signals -> impressions_last_30d, clicks_last_30d, sessions_last_30d
previous_performance: 3 signals -> impressions_prev_30d, clicks_prev_30d, sessions_prev_30d
search_performance: 3 signals -> avg_position, impressions_90d, clicks_90d

Leakage check: these current trend-derived fields will not be used as features:
['trend_direction', 'trend_pct', 'is_declining_label']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.